### Imports

In [2]:
from experiments._common import (
    H4_FCIDUMP_PATH, 
    H6_FCIDUMP_PATH,
    H4_FCI_ENERGY,
    H6_FCI_ENERGY,
    H4U_FCIDUMP_PATH,
    H4U_FCI_ENERGY
)

from qcmps import (
    FCIDumpLoader,
    GlobalQCMPS, Ansatz, HFPreset,
    Logger,                      
    GlobalOptimizer, OneSiteOptimizer, TwoSiteOptimizer, SPSAOptimizer,
    SO4Gate, BrickWallLayerType,
    Observable
)

### Logger

In [21]:
name = "h4_q1_5"

checkpoint_path = f"{name}_check.json"
logging_path = f"{name}.log"
#value_path = f"{name}.csv"

value_path = None
continue_logging = False

logger = Logger(logging_path, value_file=value_path, continue_logging=continue_logging)

### Loading the system from FCIDump

In [22]:
loader = FCIDumpLoader(H4_FCIDUMP_PATH)
loader.interleave_spin_orbitals()

hamiltonian = loader.hamiltonian
nuclear_repulsion = loader.nuclear_repulsion
hf_occupancy = loader.hf_occupancy()

num_phys_qubits = loader.n_spin_orbitals
num_bond_qubits = 1
num_qubits = num_phys_qubits + num_bond_qubits

physical_qubits = list(range(num_phys_qubits))
bond_qubits = list(range(num_phys_qubits, num_qubits))

### Building the ansatz

In [23]:
ansatz = Ansatz(num_qubits)

hf_preset = HFPreset(num_phys_qubits, hf_occupancy)
bw_layer = BrickWallLayerType(SO4Gate)
qcmps = GlobalQCMPS(num_phys_qubits, num_bond_qubits, [bw_layer])

ansatz.add_component(hf_preset, list(range(num_bond_qubits, num_qubits)))
ansatz.add_component(qcmps, list(range(num_qubits)))

### Defining the observables

In [24]:
hamiltonian = Observable(hamiltonian, physical_qubits, constant = nuclear_repulsion, label = "Total energy")
particle_number = Observable.particle_number(physical_qubits)
alpha_number = Observable.alpha_particle_number(physical_qubits)
beta_number = Observable.beta_particle_number(physical_qubits)
spin_imbalance = Observable.spin_imbalance(physical_qubits)

lambda_n = 1.0
lambda_spin = 0.1

def constrained_energy(ansatz):
    n_mean = particle_number(ansatz)
    spin_mean = spin_imbalance(ansatz)

    return (
        hamiltonian(ansatz)
        + lambda_n * (particle_number.var(ansatz) + (n_mean - 4.0) ** 2)
        + lambda_spin * (spin_imbalance.var(ansatz) + spin_mean**2)
    )

### Loading the checkpoint

In [ ]:

ansatz.load_checkpoint(checkpoint_path)

### Initial HF energy

In [25]:
energy = hamiltonian(ansatz)
logger.log(f"HF Total energy: {energy:.12f}")

HF Total energy: -2.215215410025


### Uniform parameters

In [26]:
qcmps.uniform_parameters()

energy = hamiltonian(ansatz)
logger.log(f"Initial energy: {energy:.12f}")
logger.log_value("Total energy", energy, 1)

Initial energy: -0.464844985645


### Adding layers

In [ ]:
import numpy as np

qcmps.add_layers([bw_layer], initializer = lambda n: np.random.uniform(-0.2, 0.2, size = n))
ansatz.rebuild()

energy = hamiltonian(ansatz)
logger.log(f"Energy after adding layers: {energy:.12f}")

### Perturbing the parameters

In [ ]:
qcmps.perturb_parameters(0.01)

energy = hamiltonian(ansatz)
logger.log(f"Energy after perturbing parameters: {energy:.12f}")
logger.log_value("Total energy", energy, 1)

### Global optimization

In [ ]:
logger.log("Starting Global optimization...")
global_optimizer = GlobalOptimizer(logger)
result = global_optimizer.optimize(ansatz, hamiltonian, qcmps, options = {"maxiter": 15, "maxfun": 100_000, "gtol": 1e-7, "ftol": 1e-12})

logger.log(f"Final energy: {hamiltonian(ansatz):.12f}")

### SPSA optimization

In [ ]:
logger.log("Starting SPSA optimization...")
optimizer = SPSAOptimizer(lerning_rate = 0.05, perturbation = 0.005, maxiter = 1000, seed = 42, logger = logger, log_every_n_steps = 100)
result = optimizer.optimize(ansatz, hamiltonian, qcmps)

logger.log(f"Final energy: {hamiltonian(ansatz):.12f}")

### One site optimization

In [ ]:
logger.log("Starting One site optimization...")
global_optimizer = OneSiteOptimizer(logger, log_every_n_steps = None, max_sweeps=1, local_maxiter = 10, tolerance = 1e-10)
result = global_optimizer.optimize(ansatz, hamiltonian, qcmps)

logger.log(f"Final energy: {hamiltonian(ansatz):.12f}")

### Two site optimization

In [27]:
logger.log("Starting Two site optimization...")
global_optimizer = TwoSiteOptimizer(logger, log_every_n_steps = None, max_sweeps=20, local_maxiter=10, tolerance = 1e-8)
result = global_optimizer.optimize(ansatz, hamiltonian, qcmps)

Starting Two site optimization...
[Optimizer] Sweep 1/20 started.
[Optimizer] [Sweep 1 Site pair 1-2] Total energy: -1.145486971530
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -1.148552590526
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -1.225131049427
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -1.570957254813
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -1.749651793555
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.211099464379
[Optimizer] [Sweep 1 Site pair 7-8] Total energy: -2.213041296256
[Optimizer] [Sweep 1 Site pair 6-7] Total energy: -2.213176516484
[Optimizer] [Sweep 1 Site pair 5-6] Total energy: -2.213194786304
[Optimizer] [Sweep 1 Site pair 4-5] Total energy: -2.213556232690
[Optimizer] [Sweep 1 Site pair 3-4] Total energy: -2.214616990779
[Optimizer] [Sweep 1 Site pair 2-3] Total energy: -2.215059177240
[Optimizer] [Sweep 1] Total energy: -2.215059177240, Relative change: 1.750e+00
[Optimizer] Sweep 2/20 started.
[Optimizer] [Sweep 2 Site pair

### Logging important expectations values

In [28]:
logger.log(f"Total energy: {hamiltonian(ansatz):.12f}")
logger.log(f"Total energy variance: {hamiltonian.var(ansatz):.12f}")
logger.log(f"Particle number: {particle_number(ansatz):.12f}")
logger.log(f"Particle number variance: {particle_number.var(ansatz):.12f}")
logger.log(f"Alpha particle number: {alpha_number(ansatz):.12f}")
logger.log(f"Beta particle number: {beta_number(ansatz):.12f}")

Total energy: -2.215220020890
Total energy variance: 0.069587593292
Particle number: 3.999992553119
Particle number variance: 0.000015452926
Alpha particle number: 1.999995345684
Beta particle number: 1.999997207434


### Saving the ansatz into checkpoint

In [29]:
ansatz.save_checkpoint(checkpoint_path)

### Plotting the energy

In [13]:
logger.plot_values("Total energy")
logger.plot_error("Total energy", hamiltonian(ansatz), True)

ValueError: Value file is not specified.